# Pipeline Klasifikasi Aksara Jawa

Pipeline ini mencakup:
1. Data Loading dan Preprocessing
2. Exploratory Data Analysis
3. CNN Pre-trained Model Training
4. Feature Extraction
5. Ensemble Model (KNN, Logistic Regression, SVC)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

print("="*60)
print("GPU/CUDA Configuration Check")
print("="*60)

print(f"\nTensorFlow version: {tf.__version__}")
print(f"CUDA available: {tf.test.is_built_with_cuda()}")
print(f"Built with CUDA: {tf.test.is_built_with_cuda()}")

gpus = tf.config.list_physical_devices('GPU')
print(f"\nPhysical GPUs detected: {len(gpus)}")

if gpus:
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu}")
        gpu_details = tf.config.experimental.get_device_details(gpu)
        if gpu_details:
            print(f"    Details: {gpu_details}")
    
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.list_logical_devices('GPU')
        print(f"\nLogical GPUs: {len(logical_gpus)}")
        print("GPU memory growth: ENABLED")
    except RuntimeError as e:
        print(f"Error: {e}")
    
    print(f"\nGPU device name: {tf.test.gpu_device_name()}")
    
    with tf.device('/GPU:0'):
        a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        b = tf.constant([[1.0, 2.0], [3.0, 4.0]])
        c = tf.matmul(a, b)
        print(f"\nTest computation on GPU: SUCCESS")
        print(f"Result device: {c.device}")
else:
    print("\nWARNING: No GPU detected!")
    print("Running on CPU - Training will be slower")
    print("\nTroubleshooting:")
    print("1. Check if NVIDIA GPU is available")
    print("2. Install CUDA Toolkit")
    print("3. Install cuDNN")
    print("4. Install tensorflow-gpu or tensorflow with GPU support")

print("\n" + "="*60)

## 1. Data Loading dan Preprocessing

In [ ]:
train_dir = Path('dataset-aksara-jawa/train')
val_dir = Path('dataset-aksara-jawa/val')

IMG_SIZE = 128
BATCH_SIZE = 32

def load_images_from_folder(folder_path, img_size=IMG_SIZE):
    images = []
    labels = []
    
    for class_folder in sorted(folder_path.iterdir()):
        if class_folder.is_dir():
            class_name = class_folder.name
            for img_path in class_folder.glob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        img = cv2.resize(img, (img_size, img_size))
                        img = cv2.equalizeHist(img)
                        img = cv2.GaussianBlur(img, (3, 3), 0)
                        img = img / 255.0
                        images.append(img)
                        labels.append(class_name)
    
    return np.array(images), np.array(labels)

print("Loading training data...")
X_train, y_train = load_images_from_folder(train_dir)
print(f"Training data: {X_train.shape}")

print("Loading validation data...")
X_val, y_val = load_images_from_folder(val_dir)
print(f"Validation data: {X_val.shape}")

X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

num_classes = len(label_encoder.classes_)
print(f"\nNumber of classes: {num_classes}")
print(f"Classes: {label_encoder.classes_}")

## 2. Exploratory Data Analysis

In [ ]:
train_counts = pd.Series(y_train).value_counts().sort_index()
val_counts = pd.Series(y_val).value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(train_counts.index, train_counts.values)
axes[0].set_title('Training Data Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(val_counts.index, val_counts.values)
axes[1].set_title('Validation Data Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nTraining data distribution:")
print(train_counts)
print(f"\nMean: {train_counts.mean():.2f}, Std: {train_counts.std():.2f}")

print("\nValidation data distribution:")
print(val_counts)
print(f"\nMean: {val_counts.mean():.2f}, Std: {val_counts.std():.2f}")

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 10))
axes = axes.ravel()

for idx, class_name in enumerate(label_encoder.classes_):
    class_indices = np.where(y_train == class_name)[0]
    sample_idx = class_indices[0]
    axes[idx].imshow(X_train[sample_idx].squeeze(), cmap='gray')
    axes[idx].set_title(class_name)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 3. CNN Pre-trained Model Training

In [ ]:
X_train_rgb = np.repeat(X_train, 3, axis=-1)
X_val_rgb = np.repeat(X_val, 3, axis=-1)

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1
)
datagen.fit(X_train_rgb)

In [ ]:
with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
    base_model = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    base_model.trainable = False
    
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    print(model.summary())
    print(f"\nModel created on device: {model.layers[0].weights[0].device if model.layers[0].weights else 'N/A'}")

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=10,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

print("Starting training on GPU..." if tf.config.list_physical_devices('GPU') else "Starting training on CPU...")

with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
    history = model.fit(
        datagen.flow(X_train_rgb, y_train_encoded, batch_size=BATCH_SIZE),
        validation_data=(X_val_rgb, y_val_encoded),
        epochs=50,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )

print("\nTraining completed!")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

val_loss, val_accuracy = model.evaluate(X_val_rgb, y_val_encoded, verbose=0)
print(f"\nValidation Accuracy: {val_accuracy:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

## 4. Feature Extraction

In [ ]:
feature_extractor = models.Model(
    inputs=model.input,
    outputs=model.layers[-3].output
)

print("Extracting features on GPU..." if tf.config.list_physical_devices('GPU') else "Extracting features on CPU...")

with tf.device('/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'):
    print("Extracting features from training data...")
    X_train_features = feature_extractor.predict(X_train_rgb, batch_size=BATCH_SIZE, verbose=1)
    
    print("Extracting features from validation data...")
    X_val_features = feature_extractor.predict(X_val_rgb, batch_size=BATCH_SIZE, verbose=1)

print(f"\nTraining features shape: {X_train_features.shape}")
print(f"Validation features shape: {X_val_features.shape}")

## 5. Ensemble Model Training

In [ ]:
print("Training KNN Classifier...")
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_features, y_train_encoded)
knn_pred = knn.predict(X_val_features)
knn_accuracy = accuracy_score(y_val_encoded, knn_pred)
print(f"KNN Accuracy: {knn_accuracy:.4f}")

print("\nTraining Logistic Regression...")
lr = LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)
lr.fit(X_train_features, y_train_encoded)
lr_pred = lr.predict(X_val_features)
lr_accuracy = accuracy_score(y_val_encoded, lr_pred)
print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

print("\nTraining SVC...")
svc = SVC(kernel='rbf', probability=True, random_state=42)
svc.fit(X_train_features, y_train_encoded)
svc_pred = svc.predict(X_val_features)
svc_accuracy = accuracy_score(y_val_encoded, svc_pred)
print(f"SVC Accuracy: {svc_accuracy:.4f}")

In [ ]:
print("Training Voting Classifier...")
voting_clf = VotingClassifier(
    estimators=[
        ('knn', knn),
        ('lr', lr),
        ('svc', svc)
    ],
    voting='soft',
    n_jobs=-1
)

voting_clf.fit(X_train_features, y_train_encoded)
voting_pred = voting_clf.predict(X_val_features)
voting_accuracy = accuracy_score(y_val_encoded, voting_pred)

print(f"\nEnsemble Voting Classifier Accuracy: {voting_accuracy:.4f}")

print("\n" + "="*50)
print("Model Comparison:")
print("="*50)
print(f"KNN:                 {knn_accuracy:.4f}")
print(f"Logistic Regression: {lr_accuracy:.4f}")
print(f"SVC:                 {svc_accuracy:.4f}")
print(f"Voting Ensemble:     {voting_accuracy:.4f}")
print("="*50)

## 6. Evaluation

In [ ]:
print("Classification Report for Voting Ensemble:")
print(classification_report(y_val_encoded, voting_pred, 
                          target_names=label_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_val_encoded, voting_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix - Voting Ensemble')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
accuracies = {
    'KNN': knn_accuracy,
    'Logistic Regression': lr_accuracy,
    'SVC': svc_accuracy,
    'Voting Ensemble': voting_accuracy
}

plt.figure(figsize=(10, 6))
bars = plt.bar(accuracies.keys(), accuracies.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
plt.title('Model Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()